# Pi0.5 Comprehensive Study

This notebook implements the proprioceptive state utilization study on the **Pi0.5** model,
which uses a distinct architecture from Pi0.

1. **Baseline Benchmarking**: Record unmodified performance on LIBERO, VLABench, and MetaWorld
2. **Ablation Study**: Zero state-related modules (`time_mlp_in`/`time_mlp_out`) and measure success rate drops

## Pi0.5 vs Pi0 Architecture
| Feature | Pi0 | Pi0.5 |
|---------|-----|-------|
| State Encoder | `state_proj` (Single Linear) | `time_mlp_in` / `time_mlp_out` (MLP pair) |
| Loss Function | Flow matching `F.mse_loss(u_t, v_t)` | Flow matching (same) |
| Knowledge insulation | No | Yes |

## Available Checkpoints
| Benchmark | Checkpoint | Notes |
|-----------|-----------|-------|
| LIBERO | `pi05_libero` (openpi) | Pi0.5 JAX checkpoint from GCS, served via openpi WebSocket |
| VLABench | `VLABench/pi05-primitive-10task` (HuggingFace) | Pi0.5 JAX orbax checkpoint, served via Shiduo-zh/openpi WebSocket |
| MetaWorld | `lerobot/pi05_base` (LeRobot) | Generalist pi05 — no MetaWorld fine-tuning; ~0% SR expected (negative control) |

## Expected Results
- Baseline provides reference success rates on LIBERO and VLABench with pi0.5
- Ablation shows performance impact of zeroing `time_mlp_in`/`time_mlp_out` (success rate drop)
- MetaWorld negative control confirms ~0% SR for both baseline and ablated (out-of-distribution)
- Pi0.5 replaces `state_proj` with `time_mlp_in`/`time_mlp_out` — ablation must target these

---
## Part 1 · LIBERO Benchmark with Pi0.5

Uses [Physical-Intelligence/openpi](https://github.com/Physical-Intelligence/openpi) (JAX server) for LIBERO.

| Benchmark | Checkpoint | Notes |
|-----------|-----------|-------|
| LIBERO | `pi05_libero` (openpi) | Pi0.5 JAX checkpoint from GCS, served via openpi WebSocket |

**Ablation method**:
Zero the **output** of `time_mlp_in` / `time_mlp_out` so no proprioceptive/temporal state
information reaches the expert model. `serve_policy_ablated.py` monkey-patches `Policy.__init__`
to subclass these modules and override `__call__` to return `zeros_like(output)` before the JIT
closure is compiled.

In [ ]:
# ── Paths & workspace ──────────────────────────────────────────────────────
from google.colab import drive, auth as _colab_auth
from pathlib import Path
import os, json, subprocess, sys, torch

drive.mount('/content/drive')
_colab_auth.authenticate_user()       # needed for GCS

WORKSPACE    = '/content/drive/MyDrive/pi05_study'
BASELINE_DIR = Path(f'{WORKSPACE}/Results/baseline')
ABLATION_DIR = Path(f'{WORKSPACE}/Results/ablation')
LOGS_DIR     = Path('/content/logs')

for d in [BASELINE_DIR, ABLATION_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['MUJOCO_GL'] = 'egl'
print('✅ Paths ready')

In [ ]:
# ── Install all dependencies (run once, then restart runtime) ──────────────
# Architecture:
#   LIBERO → Physical-Intelligence/openpi JAX server + Python 3.8 LIBERO venv client
#   Checkpoint: pi05_libero (pi0.5 architecture, fine-tuned on LIBERO)
import subprocess, sys, os
from pathlib import Path

# ── 1. uv (manages openpi JAX venvs) ─────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
print('✅ uv installed')

# ── 2. Physical-Intelligence/openpi (LIBERO JAX server) ──────────────────
OPENPI_DIR = '/content/openpi'
if not Path(OPENPI_DIR).exists():
    print('⏳ Cloning Physical-Intelligence/openpi (with submodules)...')
    subprocess.run(
        ['git', 'clone', '--quiet', '--recurse-submodules',
         'https://github.com/Physical-Intelligence/openpi.git', OPENPI_DIR],
        env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}, check=True)
    print('✅ openpi cloned')
else:
    print('✅ openpi already cloned')

# Sync openpi JAX server uv environment (downloads JAX, orbax, flax, etc.)
print('⏳ Syncing openpi uv environment (JAX download, ~1-2 min)...')
r = subprocess.run(
    ['uv', 'sync', '--project', OPENPI_DIR],
    capture_output=True, text=True,
    env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
if r.returncode != 0:
    print('⚠️  uv sync stderr:', r.stderr[-2000:])
    raise RuntimeError('openpi uv sync failed')
print('✅ openpi uv environment synced')

# ── 3. LIBERO client venv (Python 3.8) ───────────────────────────────────
# examples/libero/main.py runs LIBERO environments locally and calls the JAX
# server via WebSocket for policy inference.  robosuite/LIBERO require Py 3.8.
LIBERO_VENV = f'{OPENPI_DIR}/examples/libero/.venv'
LIBERO_PY   = f'{LIBERO_VENV}/bin/python'

if not Path(LIBERO_PY).exists():
    print('⏳ Creating LIBERO client venv (Python 3.8)...')
    subprocess.run(['uv', 'venv', '--python', '3.8', LIBERO_VENV], check=True)
    # Install compiled requirements (torch 1.11+cu113 from PyTorch wheel index)
    subprocess.run(
        ['uv', 'pip', 'sync',
         '--python', LIBERO_PY,
         f'{OPENPI_DIR}/examples/libero/requirements.txt',
         f'{OPENPI_DIR}/third_party/libero/requirements.txt',
         '--extra-index-url', 'https://download.pytorch.org/whl/cu113',
         '--index-strategy=unsafe-best-match'],
        check=True)
    # openpi-client WebSocket package + local LIBERO editable install
    subprocess.run(
        ['uv', 'pip', 'install', '--python', LIBERO_PY,
         '-e', f'{OPENPI_DIR}/packages/openpi-client',
         '-e', f'{OPENPI_DIR}/third_party/libero'],
        check=True)
    print('✅ LIBERO client venv ready')
else:
    print('✅ LIBERO client venv already exists')

# ── 4. Shiduo-zh/openpi (VLABench JAX server) ────────────────────────────
OPENPI_VLA = '/content/openpi_vla'
if not Path(OPENPI_VLA).exists():
    print('⏳ Cloning Shiduo-zh/openpi (VLABench fork)...')
    subprocess.run(
        ['git', 'clone', '--quiet', '--recurse-submodules',
         'https://github.com/Shiduo-zh/openpi.git', OPENPI_VLA],
        env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}, check=True)
    print('✅ Shiduo-zh/openpi cloned')
else:
    print('✅ Shiduo-zh/openpi already cloned')

print('⏳ Syncing Shiduo-zh/openpi uv environment...')
r = subprocess.run(
    ['uv', 'sync', '--project', OPENPI_VLA],
    capture_output=True, text=True,
    env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
if r.returncode != 0:
    print('⚠️  uv sync stderr:', r.stderr[-2000:])
    raise RuntimeError('Shiduo-zh/openpi uv sync failed')
print('✅ Shiduo-zh/openpi uv environment synced')

# ── HuggingFace login ──────────────────────────────────────────────────────
from huggingface_hub import login as _hf_login
_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab secrets')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')
    if _hf_token:
        print('✅ HF_TOKEN loaded from environment')
if not _hf_token:
    import getpass
    _hf_token = getpass.getpass('🔑 HuggingFace token (hidden): ')
if _hf_token:
    _hf_login(token=_hf_token, add_to_git_credential=False)
    print('✅ HuggingFace login OK')
else:
    print('⚠️  No HF token provided — may be needed for some operations.')

print('\n✅ Install complete — restart runtime, then run all cells.')

---
## 2. LIBERO Benchmark — `pi05_libero` (openpi JAX server)

Runs the **Physical-Intelligence/openpi** JAX server with the `pi05_libero` checkpoint
(pi0.5 architecture, checkpoint at `gs://openpi-assets/checkpoints/pi05_libero`).

The **LIBERO client** (`examples/libero/main.py`) runs in a dedicated Python 3.8 virtual
environment with robosuite 1.4.1 and connects to the JAX server via WebSocket.

| Suite | Tasks | Episodes per task |
|-------|-------|-------------------|
| libero_spatial | 10 | 5 |
| libero_object | 10 | 5 |
| libero_goal | 10 | 5 |
| libero_90 | 90 | 5 |

Results are logged as `"Total success rate: X.XXXXX"` by the client and collected per-suite.

### Pi0.5 Architecture Note
Pi0.5 replaces π₀'s single `state_proj` Linear layer with a pair of MLPs:
- `time_mlp_in`: Encodes temporal/state information into the model
- `time_mlp_out`: Decodes temporal/state information for action prediction

The ablation in this notebook targets these modules instead of `state_proj`.

In [ ]:
# ── LIBERO Baseline ────────────────────────────────────────────────────────
# Architecture:
#   Server: uv run scripts/serve_policy.py --env libero   (pi05_libero from GCS)
#   Client: examples/libero/.venv/bin/python examples/libero/main.py  (Py 3.8 venv)
#
# The openpi server downloads pi05_libero from GCS on first run (~5-10 min).
# The client logs "Total success rate: X.XXXXX" which we parse per-suite.
# PYTHONPATH must include third_party/libero for the LIBERO gym environments.
import os, json, re, subprocess, socket, time
from pathlib import Path

OPENPI_DIR  = '/content/openpi'
LIBERO_VENV = f'{OPENPI_DIR}/examples/libero/.venv'
LIBERO_PY   = f'{LIBERO_VENV}/bin/python'
LIBERO_MAIN = f'{OPENPI_DIR}/examples/libero/main.py'
SERVER_PORT = 8000
N_TRIALS    = 5   # trials per task  (10 tasks × 5 = 50 total per suite)

SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90']

# Start openpi LIBERO server (pi05_libero checkpoint pulled from GCS)
print('🚀 Starting openpi LIBERO server (pi05_libero from GCS, first run ~10 min)...')
server_proc = subprocess.Popen(
    ['uv', 'run', '--project', OPENPI_DIR,
     'python', f'{OPENPI_DIR}/scripts/serve_policy.py',
     '--env', 'libero',
     '--port', str(SERVER_PORT)],
    cwd=OPENPI_DIR,
    env={**os.environ, 'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85'},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f'  PID {server_proc.pid} — waiting for port {SERVER_PORT}...')

# Poll up to 10 min (GCS checkpoint download on first run)
deadline = time.time() + 600
while time.time() < deadline:
    try:
        with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
            break
    except OSError:
        time.sleep(5)
else:
    server_proc.kill()
    out, _ = server_proc.communicate()
    print('Server output:\n', out.decode()[-3000:])
    raise RuntimeError(f'openpi LIBERO server did not open port {SERVER_PORT} within 10 minutes')
print(f'✅ openpi LIBERO server ready on port {SERVER_PORT}')

# Evaluate each suite using the LIBERO venv client
libero_baseline = {}
libero_env = {**os.environ,
              'PYTHONPATH': f'{OPENPI_DIR}/third_party/libero:{os.environ.get("PYTHONPATH", "")}',
              'MUJOCO_GL': 'egl'}

for suite in SUITES:
    print(f'\n📍 Baseline: {suite}')
    result = subprocess.run(
        [LIBERO_PY, LIBERO_MAIN,
         '--task-suite-name', suite,
         '--num-trials-per-task', str(N_TRIALS),
         '--host', 'localhost',
         '--port', str(SERVER_PORT)],
        capture_output=True, text=True,
        env=libero_env,
    )
    combined = result.stdout + result.stderr
    match = re.search(r'Total success rate:\s*([\d.]+)', combined)
    if not match:
        print('Client output:\n', combined[-3000:])
        raise RuntimeError(f'Could not parse "Total success rate" for {suite}')
    sr = float(match.group(1))
    libero_baseline[suite] = {'success_rate': sr}
    print(f'  ✅ {suite} SR: {sr:.1%}')

# Stop server
server_proc.terminate()
server_proc.wait()
print('\n🛑 openpi LIBERO server stopped')

with open(BASELINE_DIR / 'libero_baseline.json', 'w') as f:
    json.dump(libero_baseline, f, indent=2)
overall = sum(r['success_rate'] for r in libero_baseline.values()) / len(libero_baseline)
print(f'✅ LIBERO baseline done  |  Overall SR: {overall:.1%}')

In [ ]:
# ── LIBERO Ablation ────────────────────────────────────────────────────────
# Writes serve_policy_ablated.py into the openpi scripts directory by prepending
# a monkey-patch that targets pi0.5-specific modules:
#   - time_mlp_in:  Encodes temporal/state information into the model
#   - time_mlp_out: Decodes temporal/state information for action prediction
#
# Pi0.5 does NOT have state_proj — it was replaced by the time_mlp pair.
# Both modules are zeroed to fully remove state encoder contribution.
import os, json, re, subprocess, socket, time
from pathlib import Path

OPENPI_DIR  = '/content/openpi'
LIBERO_VENV = f'{OPENPI_DIR}/examples/libero/.venv'
LIBERO_PY   = f'{LIBERO_VENV}/bin/python'
LIBERO_MAIN = f'{OPENPI_DIR}/examples/libero/main.py'
SERVER_PORT = 8000
N_TRIALS    = 5

SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90']

# Write ablated server script (ablation prefix + original serve_policy.py)
_serve_src = open(f'{OPENPI_DIR}/scripts/serve_policy.py').read()

_ablation_prefix = '''\
# ── ABLATION PATCH for Pi0.5 (prepended to serve_policy.py) ───────────────
# Pi0.5 replaces pi0's state_proj with time_mlp_in / time_mlp_out.
# This patch zeros the output of both modules before Policy.__init__
# compiles the JIT closure.  Policy does NOT keep a model reference
# after __init__, so we must intercept the model here.
#
# Zeroing the OUTPUT (not the weights) preserves the correct output shape
# while ensuring no state/temporal information reaches the expert model.
import openpi.policies.policy as _pol_mod
import jax.numpy as _jnp

_orig_policy_init = _pol_mod.Policy.__init__

def _find_and_patch_time_mlp(model, depth: int = 6) -> int:
    """Find and patch time_mlp_in and time_mlp_out in the pi0.5 model.
    Returns the number of modules patched (0, 1, or 2)."""
    if depth <= 0:
        return 0
    patched = 0
    for target_name in ("time_mlp_in", "time_mlp_out"):
        if hasattr(model, target_name):
            module = getattr(model, target_name)
            _orig_call = module.__class__.__call__
            module.__class__ = type(
                f"ZeroOutput_{target_name}",
                (module.__class__,),
                {"__call__": lambda self, x, _oc=_orig_call: _jnp.zeros_like(_oc(self, x))},
            )
            print(f"[ABLATION] {target_name} output zeroed (zeros_like) \\u2705")
            patched += 1
    # Also try state_proj as a fallback (in case the architecture has both)
    if hasattr(model, "state_proj"):
        sp = model.state_proj
        _orig_call = sp.__class__.__call__
        sp.__class__ = type(
            "ZeroOutputLinear",
            (sp.__class__,),
            {"__call__": lambda self, x, _oc=_orig_call: _jnp.zeros_like(_oc(self, x))},
        )
        print("[ABLATION] state_proj output zeroed (zeros_like, fallback) \\u2705")
        patched += 1
    if patched > 0:
        return patched
    # Recurse into common child attributes
    for attr in ("model", "_model", "pi0", "pi05", "transformer"):
        child = getattr(model, attr, None)
        if child is not None:
            p = _find_and_patch_time_mlp(child, depth - 1)
            if p > 0:
                return p
    return 0

def _ablated_policy_init(self, model, *args, **kwargs):
    n_patched = _find_and_patch_time_mlp(model)
    if n_patched == 0:
        print("[ABLATION] WARNING: time_mlp_in/time_mlp_out/state_proj not found!")
        print("[ABLATION] No ablation applied — results will equal baseline.")
    else:
        print(f"[ABLATION] {n_patched} module(s) patched successfully.")
    _orig_policy_init(self, model, *args, **kwargs)

_pol_mod.Policy.__init__ = _ablated_policy_init
print("[ABLATION] Policy.__init__ monkey-patched — pi0.5 state modules zeroed on load")
# ── END ABLATION PATCH ────────────────────────────────────────────────────

'''

_ablated_path = f'{OPENPI_DIR}/scripts/serve_policy_ablated_pi05.py'
with open(_ablated_path, 'w') as f:
    f.write(_ablation_prefix + _serve_src)
print(f'✅ serve_policy_ablated_pi05.py written to {_ablated_path}')

# Start ablated openpi LIBERO server (pi05_libero)
print('🚀 Starting ablated openpi LIBERO server (pi05_libero, time_mlp zeroed)...')
server_proc = subprocess.Popen(
    ['uv', 'run', '--project', OPENPI_DIR,
     'python', f'{OPENPI_DIR}/scripts/serve_policy_ablated_pi05.py',
     '--env', 'libero',
     '--port', str(SERVER_PORT)],
    cwd=OPENPI_DIR,
    env={**os.environ, 'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85'},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f'  PID {server_proc.pid} — waiting for port {SERVER_PORT}...')

deadline = time.time() + 600
while time.time() < deadline:
    try:
        with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
            break
    except OSError:
        time.sleep(5)
else:
    server_proc.kill()
    out, _ = server_proc.communicate()
    print('Server output:\n', out.decode()[-3000:])
    raise RuntimeError(f'Ablated openpi LIBERO server did not open port {SERVER_PORT} within 10 minutes')
print(f'✅ Ablated openpi LIBERO server ready on port {SERVER_PORT}')

# Run evaluation with LIBERO venv client (same as baseline)
libero_ablation = {}
libero_env = {**os.environ,
              'PYTHONPATH': f'{OPENPI_DIR}/third_party/libero:{os.environ.get("PYTHONPATH", "")}',
              'MUJOCO_GL': 'egl'}

for suite in SUITES:
    print(f'\n📍 Ablation: {suite}')
    result = subprocess.run(
        [LIBERO_PY, LIBERO_MAIN,
         '--task-suite-name', suite,
         '--num-trials-per-task', str(N_TRIALS),
         '--host', 'localhost',
         '--port', str(SERVER_PORT)],
        capture_output=True, text=True,
        env=libero_env,
    )
    combined = result.stdout + result.stderr
    match = re.search(r'Total success rate:\s*([\d.]+)', combined)
    if not match:
        print('Client output:\n', combined[-3000:])
        raise RuntimeError(f'Could not parse "Total success rate" for {suite}')
    sr = float(match.group(1))
    libero_ablation[suite] = {'success_rate': sr}
    print(f'  ✅ {suite} SR (ablated): {sr:.1%}')

# Stop server
server_proc.terminate()
server_proc.wait()
print('\n🛑 Ablated openpi LIBERO server stopped')

with open(ABLATION_DIR / 'libero_ablation.json', 'w') as f:
    json.dump(libero_ablation, f, indent=2)
print('✅ LIBERO ablation done')

---
## 2. VLABench Benchmark — `VLABench/pi05-primitive-10task`

Uses [Shiduo-zh/openpi](https://github.com/Shiduo-zh/openpi) (JAX server) for VLABench.

| Benchmark | Checkpoint | Notes |
|-----------|-----------|-------|
| VLABench | `VLABench/pi05-primitive-10task` (HuggingFace) | Pi0.5 JAX orbax checkpoint, served via Shiduo-zh/openpi WebSocket |

**Ablation method**:
Zero the **output** of `time_mlp_in` / `time_mlp_out` using the same `__call__` override
approach as the LIBERO ablation. The `serve_policy_ablated_pi05.py` patch is reused here,
injected into the Shiduo-zh/openpi `scripts/serve_policy.py`. Each module's `__call__` is
overridden to return `zeros_like(output)` before the JIT closure is compiled — the original
forward pass runs but its output is replaced with an all-zeros tensor of the correct shape.

### VLABench Tasks
VLABench evaluates 10 primitive manipulation tasks designed to test vision-language grounding:
- Tasks require language-conditioned manipulation of unseen objects
- Pi0.5's co-training on web data targets exactly this open-world generalization scenario

In [ ]:
# ── VLABench Baseline + Ablation ────────────────────────────────────────────
# Architecture:
#   Server: uv run scripts/serve_policy.py (Shiduo-zh/openpi, VLABench/pi05-primitive-10task)
#   Client: Shiduo-zh/openpi examples/vlabench/main.py
#
# The ablation reuses the same serve_policy_ablated_pi05.py already written to
# OPENPI_DIR/scripts/ by the LIBERO ablation cell. It is copied into OPENPI_VLA/scripts/.
import os, json, re, subprocess, socket, time, shutil
from pathlib import Path

OPENPI_DIR   = '/content/openpi'
OPENPI_VLA   = '/content/openpi_vla'
SERVER_PORT  = 8000
N_TRIALS     = 5
VLABENCH_TASKS = [
    'pick_apple_from_basket_and_place_on_stand',
    'pick_orange_from_shelf_and_place_on_plate',
    'pour_water_from_kettle_into_cup',
    'pick_bread_from_plate_and_place_in_basket',
    'fold_shirt',
    'pour_coffee_from_coffee_machine',
    'pick_up_purple_cup_from_rack',
    'pick_apple_from_table_and_place_in_box',
    'arrange_items_into_box',
    'pick_lemon_from_box_and_place_on_rack',
]

# Copy serve_policy_ablated_pi05.py into Shiduo-zh/openpi scripts dir
_ablated_src = Path(f'{OPENPI_DIR}/scripts/serve_policy_ablated_pi05.py')
_ablated_dst = Path(f'{OPENPI_VLA}/scripts/serve_policy_ablated_pi05.py')
if _ablated_src.exists():
    shutil.copy(_ablated_src, _ablated_dst)
    print(f'✅ serve_policy_ablated_pi05.py copied to {_ablated_dst}')
else:
    raise RuntimeError('serve_policy_ablated_pi05.py not found — run LIBERO ablation cell first')

vlabench_env = {**os.environ, 'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85'}

def run_vlabench_server(script_name: str, label: str) -> dict:
    """Start VLABench server with given script, evaluate all tasks, return results dict."""
    print(f'🚀 Starting VLABench server ({label}) with VLABench/pi05-primitive-10task...')
    server_proc = subprocess.Popen(
        ['uv', 'run', '--project', OPENPI_VLA,
         'python', f'{OPENPI_VLA}/scripts/{script_name}',
         '--env', 'vlabench',
         '--port', str(SERVER_PORT)],
        cwd=OPENPI_VLA,
        env=vlabench_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(f'  PID {server_proc.pid} — waiting for port {SERVER_PORT}...')
    deadline = time.time() + 600
    while time.time() < deadline:
        try:
            with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
                break
        except OSError:
            time.sleep(5)
    else:
        server_proc.kill()
        out, _ = server_proc.communicate()
        print('Server output:\n', out.decode()[-3000:])
        raise RuntimeError(f'VLABench server did not open port {SERVER_PORT} within 10 min')
    print(f'✅ VLABench server ready on port {SERVER_PORT}')

    results = {}
    for task in VLABENCH_TASKS:
        print(f'\n📍 {label}: {task}')
        result = subprocess.run(
            ['uv', 'run', '--project', OPENPI_VLA,
             'python', f'{OPENPI_VLA}/examples/vlabench/main.py',
             '--task-name', task,
             '--num-trials', str(N_TRIALS),
             '--host', 'localhost',
             '--port', str(SERVER_PORT)],
            capture_output=True, text=True, env=vlabench_env,
        )
        combined = result.stdout + result.stderr
        match = re.search(r'[Ss]uccess[_ ]rate[:\s]+([\d.]+)', combined)
        if not match:
            print('Client output:\n', combined[-2000:])
            raise RuntimeError(f'Could not parse success rate for {task}')
        sr = float(match.group(1))
        results[task] = {'success_rate': sr}
        print(f'  ✅ SR: {sr:.1%}')

    server_proc.terminate()
    server_proc.wait()
    print(f'\n🛑 VLABench server stopped')
    return results

# ── Baseline ─────────────────────────────────────────────────────────────────
vlabench_baseline = run_vlabench_server('serve_policy.py', 'Baseline')
with open(BASELINE_DIR / 'vlabench_baseline.json', 'w') as f:
    json.dump(vlabench_baseline, f, indent=2)
b_overall = sum(r['success_rate'] for r in vlabench_baseline.values()) / len(vlabench_baseline)
print(f'✅ VLABench baseline done  |  Overall SR: {b_overall:.1%}')

# ── Ablation ──────────────────────────────────────────────────────────────────
vlabench_ablation = run_vlabench_server('serve_policy_ablated_pi05.py', 'Ablation')
with open(ABLATION_DIR / 'vlabench_ablation.json', 'w') as f:
    json.dump(vlabench_ablation, f, indent=2)
a_overall = sum(r['success_rate'] for r in vlabench_ablation.values()) / len(vlabench_ablation)
print(f'✅ VLABench ablation done  |  Overall SR: {a_overall:.1%}')
print(f'Drop: {b_overall - a_overall:.1%}')

---
## 3. MetaWorld — Negative Control (`lerobot/pi05_base`)

| Benchmark | Checkpoint | Notes |
|-----------|-----------|-------|
| MetaWorld | `lerobot/pi05_base` (LeRobot) | Generalist pi05 — no MetaWorld fine-tuning; ~0% SR expected |

> **Negative Control**: `pi05_base` is a generalist model never fine-tuned on MetaWorld tasks.
> Including this section confirms that: (a) the evaluation pipeline works correctly, and (b)
> without task-specific training, pi05 has no MetaWorld capability — consistent with pi0.

**Ablation target**: `time_mlp_in` / `time_mlp_out` via PyTorch `register_forward_hook`.
Since pi0.5 does **not** have `state_proj`, the hook targets the MLP pair instead.
With ~0% baseline SR, both baseline and ablated SR will be near 0 — confirming the
negative control status rather than measuring state utilization.

In [ ]:
# ── MetaWorld Negative Control ────────────────────────────────────────────────
# Uses lerobot/pi05_base (PI05Policy) — generalist, ~0% SR on MetaWorld expected.
# Mirrors the pi0_complete.ipynb MetaWorld section.
import os, json, subprocess, sys
from pathlib import Path

POLICY_ID    = 'lerobot/pi05_base'
N_TRIALS_MW  = 5
MW_TASKS     = ['reach-v2', 'push-v2', 'pick-place-v2', 'door-open-v2', 'drawer-close-v2']

_mw_eval_pi05 = '''\
#!/usr/bin/env python3
"""MetaWorld evaluation for Pi0.5 (negative control — generalist lerobot/pi05_base)."""
import argparse, json, sys
import numpy as np

parser = argparse.ArgumentParser()
parser.add_argument('--policy', default='lerobot/pi05_base')
parser.add_argument('--tasks', nargs='+',
                    default=['reach-v2','push-v2','pick-place-v2','door-open-v2','drawer-close-v2'])
parser.add_argument('--n-trials', type=int, default=5)
parser.add_argument('--output-path', default='mw_pi05_results.json')
parser.add_argument('--ablate', action='store_true',
                    help='Zero time_mlp_in/time_mlp_out output via register_forward_hook')
args = parser.parse_args()

import torch
from lerobot.policies.pi05 import PI05Policy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
policy = PI05Policy.from_pretrained(args.policy).to(device).eval()

if args.ablate:
    def _zero_hook(m, i, o):
        return torch.zeros_like(o)
    _handles = []
    for name in ('time_mlp_in', 'time_mlp_out'):
        mod = None
        for n, m in policy.named_modules():
            if n.endswith(name) or n == name:
                mod = m
                break
        if mod is not None:
            _handles.append(mod.register_forward_hook(_zero_hook))
            print(f"[ABLATION] {name} output zeroed \u2705")
        else:
            print(f"[ABLATION] WARNING: {name} not found — check PI05Policy attribute names")

import metaworld
import random
results = {}
for task_name in args.tasks:
    ml1 = metaworld.ML1(task_name)
    env_cls = ml1.train_classes[task_name]
    task = random.choice(ml1.train_tasks)
    env = env_cls()
    env.set_task(task)
    successes = []
    for _ in range(args.n_trials):
        obs, _ = env.reset()
        done = False
        success = False
        for _step in range(500):
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                action = policy(obs_t).squeeze(0).cpu().numpy()
            obs, reward, terminated, truncated, info = env.step(action)
            if info.get("success", False):
                success = True
            if terminated or truncated:
                break
        successes.append(float(success))
    sr = float(np.mean(successes))
    results[task_name] = {"success_rate": sr}
    print(f"{task_name}: {sr:.1%}")

overall = float(np.mean([v["success_rate"] for v in results.values()]))
results["overall"] = {"success_rate": overall}
print(f"\\nOverall: {overall:.1%}")
with open(args.output_path, "w") as f:
    json.dump(results, f, indent=2)
'''

Path('/content/mw_eval_pi05.py').write_text(_mw_eval_pi05)
print('✅ mw_eval_pi05.py written to /content/mw_eval_pi05.py')

mw_baseline = {}
mw_ablation = {}

# Baseline
print('\n📍 MetaWorld baseline (lerobot/pi05_base — negative control)...')
r = subprocess.run(
    [sys.executable, '/content/mw_eval_pi05.py',
     '--policy', POLICY_ID,
     '--n-trials', str(N_TRIALS_MW),
     '--output-path', '/content/mw_pi05_baseline.json'],
    capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
mw_baseline = json.loads(Path('/content/mw_pi05_baseline.json').read_text())
with open(BASELINE_DIR / 'metaworld_baseline.json', 'w') as f:
    json.dump(mw_baseline, f, indent=2)
print(f'✅ MetaWorld baseline done  |  Overall: {mw_baseline.get("overall", {}).get("success_rate", float("nan")):.1%}')

# Ablation
print('\n📍 MetaWorld ablation (time_mlp_in/time_mlp_out zeroed)...')
r = subprocess.run(
    [sys.executable, '/content/mw_eval_pi05.py',
     '--policy', POLICY_ID,
     '--n-trials', str(N_TRIALS_MW),
     '--output-path', '/content/mw_pi05_ablation.json',
     '--ablate'],
    capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
mw_ablation = json.loads(Path('/content/mw_pi05_ablation.json').read_text())
with open(ABLATION_DIR / 'metaworld_ablation.json', 'w') as f:
    json.dump(mw_ablation, f, indent=2)
print(f'✅ MetaWorld ablation done  |  Overall: {mw_ablation.get("overall", {}).get("success_rate", float("nan")):.1%}')
print(f'   (Expected ~0% for both — negative control)')

In [ ]:
# ── Results Summary ───────────────────────────────────────────────────────
import json
from pathlib import Path

def load_json(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else {}

libero_base = load_json(BASELINE_DIR / 'libero_baseline.json')
libero_abl  = load_json(ABLATION_DIR / 'libero_ablation.json')

def mean_sr(d):
    rates = [v['success_rate'] for v in d.values() if 'success_rate' in v]
    return sum(rates)/len(rates) if rates else float('nan')

print('='*65)
print(f'{"Suite":<20} {"Baseline SR":>12} {"Ablated SR":>12} {"Drop":>8}')
print('='*65)
for suite in ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90']:
    b = libero_base.get(suite, {}).get('success_rate', float('nan'))
    a = libero_abl.get(suite, {}).get('success_rate', float('nan'))
    drop = b - a if not (b != b or a != a) else float('nan')  # nan check
    print(f'{suite:<20} {b:>11.1%} {a:>11.1%} {drop:>7.1%}')
print('='*65)
b_overall = mean_sr(libero_base)
a_overall = mean_sr(libero_abl)
print(f'{"OVERALL":<20} {b_overall:>11.1%} {a_overall:>11.1%} {b_overall - a_overall:>7.1%}')
print('='*65)
print()
print('Model: pi0.5 (pi05_libero checkpoint)')
print('Ablation: time_mlp_in/time_mlp_out output zeroed')
print('Low drop → state encoder underutilized (paper hypothesis ✅)')

---
## Notes

### Why no PyTorch gradient analysis yet?
Pi0.5 now has a PyTorch checkpoint (`lerobot/pi05_libero_finetuned` via `PI05Policy`).
Gradient analysis using `torch.autograd` is now possible and is left as future work —
see `pi05_hooks.py` for the hook implementation targeting `time_mlp_in`/`time_mlp_out`.

### Comparison with pi0_complete.ipynb
| Notebook | LIBERO Checkpoint | State Module | MetaWorld | VLABench | Gradient Analysis |
|----------|------------------|--------------|-----------|----------|-------------------|
| `pi0_complete.ipynb` | `lerobot/pi0_libero_finetuned` | `state_proj` | ✅ `lerobot/pi0_base` (neg. ctrl) | ✅ `VLABench/pi0-primitive-10task` | ✅ PyTorch |
| `pi05_complete.ipynb` | `pi05_libero` (openpi GCS) | `time_mlp_in`/`time_mlp_out` | ✅ `lerobot/pi05_base` (neg. ctrl) | ✅ `VLABench/pi05-primitive-10task` | ⏳ Available via `lerobot/pi05_libero_finetuned` (future work) |

### Checkpoint provenance
| Checkpoint | Source | Type |
|-----------|--------|------|
| `pi05_libero` (GCS) | Physical Intelligence / openpi | JAX orbax |
| `lerobot/pi05_libero_finetuned` | HuggingFace LeRobot | PyTorch |
| `lerobot/pi05_base` | HuggingFace LeRobot | PyTorch |
| `VLABench/pi05-primitive-10task` | HuggingFace VLABench | JAX orbax |